# BP5 Gate 2 — Data Verification & Driver-Feature Engineering
**Customer360 Navigator Enterprise Suite — Root-Cause & Driver Analytics**

## Why this notebook exists, and how it differs from BP1-4's Gate 2

BP5 has no single supervised target and integrates no BANKING77 taxonomy (Master Plan BP table:
Integrates BANKING77? = NO). Per BP5 Gate 1's own real, live-run policy.json
(`generated_at_utc: 2026-09-24T07:16:13Z`), BP5 tests real candidate driver fields (Product,
Sub-product, Issue, Sub-issue, Submitted via, Company; State as a control field only) for
statistical **association** — never causation — against **two** real, distinct CFPB outcome
fields:

- `outcome_1_intervention_required` — reused verbatim from BP3 Gate 1's own real target
  definition on `Company response to consumer`.
- `outcome_2_timely_response_failure` — new to BP5: binary from the real `Timely response?`
  field, computed over the full real population independently of `Company response to consumer`.

Gate 2's job here is therefore doubled relative to BP1-4: tag every real row with **both**
outcome fields (each with its own exclusion reason), apply the explicit null-sentinel fill to the
real candidate driver columns that have real nulls, build a driver-field lineage table
(distinguishing PRIMARY drivers, the CONTROL field, and BARRED columns), re-verify BP5 Gate 1's
own `outcome_field_overlap_live_check` against the real data again (this relationship is central
to BP5's own leakage_rules), and write the Gold layer — mirroring BP3 Gate 2's
"every row tagged, none dropped" pattern.

## Standing rules this notebook follows

- **WARP**: Polars lazy scans throughout, category dtypes reused from `taxonomy_mapper.CFPB_DTYPES`
  (not re-derived), no pandas, no eager full-file loads.
- **Zero-fabrication / live drift checks**: every distribution, null count, class balance, and
  overlap figure in this notebook is re-measured live against the real CFPB file and cross-checked
  against BP5 Gate 1's own real, live-run policy.json — never trusted from the artifact alone.
- **Execution boundary**: this notebook is delivered as source only. It has never been executed
  by Claude — only the user runs real pipeline notebooks, per this project's standing rule. Every
  number, check, and Gold-layer output referenced above as "real" was independently reproduced in
  a disposable sandbox against a real staged copy of the CFPB data as a pre-delivery verification
  step (not a substitute for the user's own real run, and no sandbox output is delivered) — see
  the Evidence Ledger entry dated 2026-09-24 for the full verification record, including one bug
  (`pct_intervention_required_rows_also_timely_no` using the wrong denominator) caught and fixed
  by that sandbox cross-check before this notebook was ever delivered.

## Prerequisite

BP5 Gate 1 must be real-run at least once (`configs/bp5_root_cause_driver_analytics.yaml` front
matter and `notebooks/bp5_root_cause_driver_analytics/artifacts/policy.json` must both exist) —
this notebook raises `FileNotFoundError` immediately if either is missing, rather than fabricating
what Gate 1 would have produced.

## What this notebook does NOT do

- No model is trained and no champion is selected here — that is Gate 3's job
  (hypothesis testing / logistic regression / SHAP, per BP5 Gate 1's own `methodology_policy`).
- No causal claim is made anywhere in this notebook — every finding downstream is reported as a
  statistical association only, per BP5 Gate 1's own `association_not_causation_disclaimer`.
- `State` is prepared identically to the other candidate fields (one-hot, null-sentinel-filled)
  but is never described as a "driver" anywhere in this notebook's own output — it is a control
  field only, per BP5 Gate 1's own explicit scoping choice.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP5 Gate 2 data verification / driver-feature engineering
notebook. Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import polars as pl
from IPython.display import display

from taxonomy.taxonomy_mapper import CFPB_DTYPES
from features.bp5_driver_features import (
    BARRED_COLUMNS,
    FEATURE_COLS_CATEGORICAL,
    COMPANY_COL,
    CONTROL_FIELD,
    null_screen_report,
    load_cfpb_with_outcomes_and_filled_features,
    outcome_overlap_report,
    feature_lineage_table,
    build_bp5_driver_gold_layer,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp5_root_cause_driver_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CFPB_PATH = DATA_RAW / "cfpb_complaints.csv"
BP5_CONFIG_PATH = CONFIGS_DIR / "bp5_root_cause_driver_analytics.yaml"
POLICY_JSON_PATH = ARTIFACTS_DIR / "policy.json"

for p in (CFPB_PATH, BP5_CONFIG_PATH, POLICY_JSON_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP5 Gate 1 has been real-run at least once "
            "(the config front matter and policy.json artifact are both prerequisites)."
        )

with open(POLICY_JSON_PATH, "r", encoding="utf-8") as f:
    gate1_policy = json.load(f)
print(f"[OK] Loaded Gate 1 policy.json (generated_at_utc={gate1_policy['generated_at_utc']}).")

# ============================================================
# SECTION 4: LIVE drift check - re-measure 'Company response to consumer' and 'Timely response?'
# against what Gate 1's policy.json recorded. Zero-fabrication: never trust the artifact's
# baked-in counts without re-measuring against the real file. Both real fields matter here since
# BOTH define one of BP5's two outcomes.
# ============================================================
cfpb_lazy_base = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)

live_company_response_counts = (
    cfpb_lazy_base.group_by("Company response to consumer").agg(pl.len().alias("row_count")).collect()
)
live_company_response_dict = dict(
    zip(
        live_company_response_counts["Company response to consumer"].cast(pl.Utf8).to_list(),
        live_company_response_counts["row_count"].to_list(),
    )
)
documented_company_response_dict = {
    d["Company response to consumer"]: d["n"] for d in gate1_policy["live_checks"]["company_response_to_consumer_distribution"]
}
company_response_drift = {
    k: {"documented": documented_company_response_dict.get(k), "live": live_company_response_dict.get(k)}
    for k in set(documented_company_response_dict) | set(live_company_response_dict)
    if documented_company_response_dict.get(k) != live_company_response_dict.get(k)
}

live_timely_response_counts = (
    cfpb_lazy_base.group_by("Timely response?").agg(pl.len().alias("row_count")).collect()
)
live_timely_response_dict = dict(
    zip(
        live_timely_response_counts["Timely response?"].cast(pl.Utf8).to_list(),
        live_timely_response_counts["row_count"].to_list(),
    )
)
documented_timely_response_dict = {
    d["Timely response?"]: d["n"] for d in gate1_policy["live_checks"]["timely_response_distribution"]
}
timely_response_drift = {
    k: {"documented": documented_timely_response_dict.get(k), "live": live_timely_response_dict.get(k)}
    for k in set(documented_timely_response_dict) | set(live_timely_response_dict)
    if documented_timely_response_dict.get(k) != live_timely_response_dict.get(k)
}

if company_response_drift or timely_response_drift:
    print(f"[DRIFT DETECTED] company_response_drift={json.dumps(company_response_drift, indent=2)}")
    print(f"[DRIFT DETECTED] timely_response_drift={json.dumps(timely_response_drift, indent=2)}")
else:
    print(
        "[OK] Live 'Company response to consumer' and 'Timely response?' distributions both match "
        "Gate 1's policy.json exactly - no drift since BP5 Gate 1's real run."
    )

# ============================================================
# SECTION 5: LIVE null screen on the 7 real candidate driver / control columns - compared against
# Gate 1's policy.json candidate_driver_field_profile. Gate 2's own exit criterion: zero nulls
# silently dropped - this section measures the real numbers that criterion is checked against.
# ============================================================
live_null_report = null_screen_report(cfpb_lazy_base)
print("\n=== LIVE NULL SCREEN: BP5 candidate driver / control columns ===")
display(live_null_report.to_pandas())

gate1_driver_profile = gate1_policy["live_checks"]["candidate_driver_field_profile"]
null_drift = {
    row["column"]: {"documented": gate1_driver_profile.get(row["column"], {}).get("n_null"), "live": row["null_count"]}
    for row in live_null_report.to_dicts()
    if gate1_driver_profile.get(row["column"], {}).get("n_null") != row["null_count"]
}
if null_drift:
    print(f"[DRIFT DETECTED] null_count_drift={json.dumps(null_drift, indent=2)}")
else:
    print(
        "[OK] Live null counts for every BP5 candidate driver / control column match Gate 1's "
        "policy.json exactly - no drift."
    )

# ============================================================
# SECTION 6: Live column-list check for the narrative-text/PII compliance touchpoint (Master Plan
# Section 8, Gate 2 row) - confirms live, not assumed, that no narrative-text column exists in
# this real extract, matching BP1-4's own Gate 2 precedent.
# ============================================================
live_columns = cfpb_lazy_base.collect_schema().names()
NARRATIVE_TEXT_KEYWORDS = ("narrative", "complaint text", "free text", "consumer complaint narrative")
narrative_columns_found = [c for c in live_columns if any(kw in c.lower() for kw in NARRATIVE_TEXT_KEYWORDS)]
print(f"\n[COMPLIANCE CHECK] Real CFPB columns (live): {live_columns}")
if narrative_columns_found:
    print(
        f"[ACTION REQUIRED] Narrative-text column(s) found: {narrative_columns_found} - a PII "
        "screen is required before this data can be used downstream. NOT performed by this "
        "notebook."
    )
else:
    print(
        "[OK] No narrative-text column present in this real 15-column CFPB extract (live-"
        "confirmed) - the PII-screen-on-narrative-text compliance touchpoint has no narrative "
        "text to screen for BP5, matching BP1-4's own earlier finding."
    )

# ============================================================
# SECTION 7: Tag every real row with BOTH outcome fields + their exclusion reasons, and cross-
# check the live result against Gate 1's recorded class balances for each outcome independently.
# ============================================================
cfpb_lazy_tagged = load_cfpb_with_outcomes_and_filled_features(CFPB_PATH)

live_outcome_1_balance = (
    cfpb_lazy_tagged.group_by(["outcome_1_intervention_required", "outcome_1_exclusion_reason"])
    .agg(pl.len().alias("row_count"))
    .collect()
    .sort("row_count", descending=True)
)
print("\n=== LIVE OUTCOME 1 TAGGING: outcome_1_intervention_required x outcome_1_exclusion_reason ===")
display(live_outcome_1_balance.to_pandas())

n_o1_positive = int(live_outcome_1_balance.filter(pl.col("outcome_1_intervention_required") == 1)["row_count"].sum())
n_o1_negative = int(live_outcome_1_balance.filter(pl.col("outcome_1_intervention_required") == 0)["row_count"].sum())
n_o1_excluded_pending = int(
    live_outcome_1_balance.filter(pl.col("outcome_1_exclusion_reason") == "EXCLUDED_PENDING")["row_count"].sum()
)
n_o1_excluded_untimely = int(
    live_outcome_1_balance.filter(
        pl.col("outcome_1_exclusion_reason") == "EXCLUDED_UNTIMELY_RESPONSE_OVERLAPS_BP2"
    )["row_count"].sum()
)
n_o1_excluded_null = int(
    live_outcome_1_balance.filter(pl.col("outcome_1_exclusion_reason") == "EXCLUDED_UNKNOWN_NULL_RESPONSE")[
        "row_count"
    ].sum()
)
n_o1_trainable = n_o1_positive + n_o1_negative
n_o1_excluded_total = n_o1_excluded_pending + n_o1_excluded_untimely + n_o1_excluded_null

gate1_o1_balance = gate1_policy["live_checks"]["outcome_1_intervention_required_class_balance"]
outcome_1_matches_gate1 = (
    n_o1_positive == gate1_o1_balance["n_intervention_required"]
    and n_o1_negative == gate1_o1_balance["n_no_intervention_required"]
    and n_o1_excluded_pending == gate1_o1_balance["n_excluded_in_progress"]
    and n_o1_excluded_untimely == gate1_o1_balance["n_excluded_untimely_response"]
    and n_o1_excluded_null == gate1_o1_balance["n_excluded_null_response"]
)
print(
    f"\n[FINDING] outcome_1: {n_o1_positive:,} positive, {n_o1_negative:,} negative, "
    f"{n_o1_trainable:,} trainable, {n_o1_excluded_total:,} excluded. Matches Gate 1's recorded "
    f"class balance exactly: {outcome_1_matches_gate1}"
)

live_outcome_2_balance = (
    cfpb_lazy_tagged.group_by(["outcome_2_timely_response_failure", "outcome_2_exclusion_reason"])
    .agg(pl.len().alias("row_count"))
    .collect()
    .sort("row_count", descending=True)
)
print("\n=== LIVE OUTCOME 2 TAGGING: outcome_2_timely_response_failure x outcome_2_exclusion_reason ===")
display(live_outcome_2_balance.to_pandas())

n_o2_positive = int(
    live_outcome_2_balance.filter(pl.col("outcome_2_timely_response_failure") == 1)["row_count"].sum()
)
n_o2_negative = int(
    live_outcome_2_balance.filter(pl.col("outcome_2_timely_response_failure") == 0)["row_count"].sum()
)
n_o2_excluded_null = int(
    live_outcome_2_balance.filter(pl.col("outcome_2_exclusion_reason") == "EXCLUDED_NULL_TIMELY_RESPONSE")[
        "row_count"
    ].sum()
)
n_o2_trainable = n_o2_positive + n_o2_negative

gate1_o2_balance = gate1_policy["live_checks"]["outcome_2_timely_response_failure_class_balance"]
outcome_2_matches_gate1 = (
    n_o2_positive == gate1_o2_balance["n_timely_response_failure"]
    and n_o2_negative == gate1_o2_balance["n_timely_response_ok"]
    and n_o2_excluded_null == gate1_o2_balance["n_excluded_null_timely_response"]
)
print(
    f"\n[FINDING] outcome_2: {n_o2_positive:,} positive, {n_o2_negative:,} negative, "
    f"{n_o2_trainable:,} trainable, {n_o2_excluded_null:,} excluded (null). Matches Gate 1's "
    f"recorded class balance exactly: {outcome_2_matches_gate1}"
)

# ============================================================
# SECTION 8: Live re-verification of the outcome-field overlap check (BP5's own leakage_rules
# depend on this real relationship, not assumed) - cross-checked against Gate 1's recorded numbers.
# ============================================================
live_overlap = outcome_overlap_report(cfpb_lazy_tagged)
gate1_overlap = gate1_policy["live_checks"]["outcome_field_overlap_live_check"]
overlap_matches_gate1 = all(live_overlap[k] == gate1_overlap[k] for k in gate1_overlap)
print("\n=== LIVE OUTCOME-FIELD OVERLAP RE-VERIFICATION ===")
print(json.dumps(live_overlap, indent=2))
print(f"[FINDING] Matches Gate 1's recorded overlap check exactly: {overlap_matches_gate1}")

# ============================================================
# SECTION 9: Build and save the driver-field lineage table (Gate 2's own named exit-criterion
# artifact)
# ============================================================
lineage = feature_lineage_table()
print("\n=== DRIVER-FIELD LINEAGE TABLE ===")
display(lineage.to_pandas())

lineage_path = ARTIFACTS_DIR / "gate2_driver_field_lineage.csv"
lineage.write_csv(lineage_path)
print(f"[SAVED] {lineage_path.relative_to(PROJECT_ROOT)}")

# A barred column is allowed to appear in the lineage table as EITHER "(none - barred)" (the
# explicit exclusion row) OR one of the two outcome TARGET rows (source_column == the field that
# defines that outcome - it is the LABEL, never an input driver field, tagged "TARGET" in its own
# driver_field string precisely so this check can tell the legitimate cases apart from a real leak).
barred_in_lineage_as_driver = lineage.filter(
    pl.col("source_column").is_in(BARRED_COLUMNS)
    & (pl.col("driver_field") != "(none - barred)")
    & (~pl.col("driver_field").str.contains("TARGET"))
)
no_barred_column_used_as_driver = barred_in_lineage_as_driver.height == 0

# ============================================================
# SECTION 10: Write the BP5 CFPB-with-both-outcomes Gold layer (Parquet, WARP)
# ============================================================
gold_summary = build_bp5_driver_gold_layer(cfpb_lazy_tagged, DATA_PROCESSED)
print(
    f"\n[SAVED] BP5 Root-Cause Driver Gold: {gold_summary['cfpb_driver_gold_path']} "
    f"({gold_summary['cfpb_driver_gold_rows_written']} rows)"
)

# ============================================================
# SECTION 11: Write the Gate 2 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified, fifth BP to do so)
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

gate2_marker = (
    "# --- Gate 2 (Data Verification & Feature Engineering) results (appended, idempotent overwrite) ---"
)
gate2_block_lines = (
    [
        f"drift_vs_gate1_company_response_live_check: {'none' if not company_response_drift else 'DRIFT_DETECTED'}",
        f"drift_vs_gate1_timely_response_live_check: {'none' if not timely_response_drift else 'DRIFT_DETECTED'}",
        f"null_count_drift_vs_gate1_policy_json: {'none' if not null_drift else 'DRIFT_DETECTED'}",
        f"narrative_text_column_found: {bool(narrative_columns_found)}",
        "candidate_driver_null_counts:",
    ]
    + [f"  {row['column']}: {row['null_count']}" for row in live_null_report.to_dicts()]
    + [
        f"n_outcome_1_positive: {n_o1_positive}",
        f"n_outcome_1_negative: {n_o1_negative}",
        f"n_outcome_1_trainable: {n_o1_trainable}",
        f"n_outcome_1_excluded_total: {n_o1_excluded_total}",
        f"outcome_1_matches_gate1_policy_json: {outcome_1_matches_gate1}",
        f"n_outcome_2_positive: {n_o2_positive}",
        f"n_outcome_2_negative: {n_o2_negative}",
        f"n_outcome_2_trainable: {n_o2_trainable}",
        f"n_outcome_2_excluded_null: {n_o2_excluded_null}",
        f"outcome_2_matches_gate1_policy_json: {outcome_2_matches_gate1}",
        f"outcome_overlap_matches_gate1_policy_json: {overlap_matches_gate1}",
        f'driver_field_lineage_path: "{lineage_path.relative_to(PROJECT_ROOT).as_posix()}"',
        f"no_barred_column_used_as_driver: {no_barred_column_used_as_driver}",
        'cfpb_driver_gold_path: "'
        + Path(gold_summary["cfpb_driver_gold_path"]).relative_to(PROJECT_ROOT).as_posix()
        + '"',
        f"cfpb_driver_gold_rows_written: {gold_summary['cfpb_driver_gold_rows_written']}",
    ]
)
write_gate_block(BP5_CONFIG_PATH, gate2_marker, gate2_block_lines)
print(f"[SAVED] gate2 block written to {BP5_CONFIG_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "no_drift_vs_gate1_company_response_distribution": not company_response_drift,
    "no_drift_vs_gate1_timely_response_distribution": not timely_response_drift,
    "no_null_count_drift_vs_gate1_policy_json": not null_drift,
    "no_narrative_text_column_present": not narrative_columns_found,
    "outcome_1_tagging_matches_gate1_policy_json": outcome_1_matches_gate1,
    "outcome_2_tagging_matches_gate1_policy_json": outcome_2_matches_gate1,
    "outcome_overlap_matches_gate1_policy_json": overlap_matches_gate1,
    "outcome_1_trainable_plus_excluded_equals_total": (
        n_o1_trainable + n_o1_excluded_total == gate1_policy["live_checks"]["cfpb_row_count"]
    ),
    "outcome_2_trainable_plus_excluded_equals_total": (
        n_o2_trainable + n_o2_excluded_null == gate1_policy["live_checks"]["cfpb_row_count"]
    ),
    "driver_field_lineage_csv_written": lineage_path.exists(),
    "no_barred_column_used_as_driver_field": no_barred_column_used_as_driver,
    "gold_layer_row_count_matches_source": (
        gold_summary["cfpb_driver_gold_rows_written"] == gate1_policy["live_checks"]["cfpb_row_count"]
    ),
    "bp5_config_gate2_block_written": BP5_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP5 Gate 2 complete - every real row tagged with both "
    f"outcome_1_intervention_required ({n_o1_trainable:,} trainable) and "
    f"outcome_2_timely_response_failure ({n_o2_trainable:,} trainable), none dropped, zero nulls "
    "silently dropped in any candidate driver / control column, outcome-field overlap "
    "re-verified against Gate 1, driver-field lineage table written. Proceed to BP5 Gate 3 "
    "(Hypothesis Testing / Regression / SHAP Association Benchmark) next."
)
